In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Hey clientes dataset

In [5]:
# Load the dataset
df = pd.read_csv('../../Data/dataset_transacciones/hey_clientes.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15025 entries, 0 to 15024
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  15025 non-null  object 
 1   edad                     15025 non-null  int64  
 2   sexo                     15025 non-null  object 
 3   estado                   14593 non-null  object 
 4   ciudad                   14593 non-null  object 
 5   nivel_educativo          15025 non-null  object 
 6   ocupacion                15025 non-null  object 
 7   ingreso_mensual_mxn      15025 non-null  int64  
 8   antiguedad_dias          15025 non-null  int64  
 9   es_hey_pro               15025 non-null  bool   
 10  nomina_domiciliada       15025 non-null  bool   
 11  canal_apertura           15025 non-null  object 
 12  score_buro               15025 non-null  int64  
 13  dias_desde_ultimo_login  15025 non-null  int64  
 14  preferencia_canal     

## 1. Age Band Feature

Bins de `edad`: [18-25, 26-35, 36-50, 51+]

In [6]:
def create_age_band(edad):
    if 18 <= edad <= 25:
        return '18-25'
    elif 26 <= edad <= 35:
        return '26-35'
    elif 36 <= edad <= 50:
        return '36-50'
    elif edad >= 51:
        return '51+'
    else:
        return None

df['age_band'] = df['edad'].apply(create_age_band)
print(df[['edad', 'age_band']].head())

   edad age_band
0    21    18-25
1    18    18-25
2    23    18-25
3    32    26-35
4    26    26-35


## 2. Income Tier Feature
Quantile-bin `ingreso_mensual_mx` en tiers Q1-Q4.

In [7]:
df['income_tier'] = pd.qcut(df['ingreso_mensual_mxn'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print(df[['ingreso_mensual_mxn', 'income_tier']].head())

   ingreso_mensual_mxn income_tier
0                24500          Q3
1                19000          Q2
2                14000          Q1
3                61000          Q4
4                27000          Q3


## 3. Engagement Score Feature
Normalizar y combinar: `dias_desde_ultimo_login`, `satisfaccion_1_10`, `es_hey_pro`, `nomina_domiciliada`.

In [8]:
# Normalize components
scaler = MinMaxScaler()

# Inverse of days since last login (higher frequency = higher engagement)
df['login_freq'] = 1 / (df['dias_desde_ultimo_login'] + 1)  # +1 to avoid division by zero
df['login_freq_norm'] = scaler.fit_transform(df[['login_freq']])

# Normalize satisfaction (already 1-10, but scale to 0-1)
df['satisfaccion_norm'] = (df['satisfaccion_1_10'] - 1) / 9  # Min 1, max 10

# Binary features to 0-1
df['es_hey_pro_num'] = df['es_hey_pro'].astype(int)
df['nomina_domiciliada_num'] = df['nomina_domiciliada'].astype(int)

# Combine into engagement score (average of normalized components)
df['engagement_score'] = (
    df['login_freq_norm'] + 
    df['satisfaccion_norm'] + 
    df['es_hey_pro_num'] + 
    df['nomina_domiciliada_num']
) / 4

print(df[['dias_desde_ultimo_login', 'satisfaccion_1_10', 'es_hey_pro', 'nomina_domiciliada', 'engagement_score']].head())

   dias_desde_ultimo_login  satisfaccion_1_10  es_hey_pro  nomina_domiciliada  \
0                        1               10.0        True               False   
1                        3                8.0        True               False   
2                        3                8.0        True               False   
3                       16               10.0       False               False   
4                        1                7.0        True               False   

   engagement_score  
0          0.624306  
1          0.505903  
2          0.505903  
3          0.263399  
4          0.540972  


## 4. Credit Health Feature
La neta hay que quedarse con el score buro

## 5. Digital Maturity Feature
Combinar `canal_apertura`, `preferencia_canal`, y la inversa de `dias_desde_ultimo_login`.

In [9]:
# Map canal_apertura to score (App = 1, Fan Shop = 0)
df['canal_score'] = df['canal_apertura'].map({'App': 1, 'Fan Shop': 0})

# Map preferencia_canal to scores (assuming app_ios/android/huawei are similar)
df['preferencia_score'] = df['preferencia_canal'].map({
    'app_ios': 1, 
    'app_android': 1, 
    'app_huawei': 1
}).fillna(0)  # If other values, assume 0

# Use the same login_freq_norm from engagement score

# Combine into digital maturity score
df['digital_maturity'] = (
    df['canal_score'] + 
    df['preferencia_score'] + 
    df['login_freq_norm']
) / 3

print(df[['canal_apertura', 'preferencia_canal', 'dias_desde_ultimo_login', 'digital_maturity']].head())

  canal_apertura preferencia_canal  dias_desde_ultimo_login  digital_maturity
0            App       app_android                        1          0.832407
1            App       app_android                        3          0.748611
2            App           app_ios                        3          0.748611
3       Fan Shop           app_ios                       16          0.351198
4       Fan Shop           app_ios                        1          0.499074


## 6. Tenure Band Feature
Mejor quedarse con los días de antiguedad

## 7. Vulnerability Flag Feature
Flag basada en `recibe_remesas`, `ocupacion==Desempleado`, y un bajo estrato de ingreso (Q1).

In [10]:
df['vulnerability_flag'] = (
    df['recibe_remesas'] | 
    (df['ocupacion'] == 'Desempleado') | 
    (df['income_tier'] == 'Q1')
).astype(int)

print(df[['recibe_remesas', 'ocupacion', 'income_tier', 'vulnerability_flag']].head())

   recibe_remesas   ocupacion income_tier  vulnerability_flag
0           False    Empleado          Q3                   0
1           False  Estudiante          Q2                   0
2           False  Estudiante          Q1                   1
3            True    Empleado          Q4                   1
4           False  Empresario          Q3                   0


## 8. Financial Sophistication Feature
Combinar `nivel_educativo`, `usa_hey_shop` (Como un proxy de la actividad de inversion), y `score_buro`.

In [12]:
# Map education level to scores
education_scores = {
    'Secundaria': 1,
    'Preparatoria': 2,
    'Licenciatura': 3,
    'Posgrado': 4
}
df['education_score'] = df['nivel_educativo'].map(education_scores)
df['education_score_norm'] = scaler.fit_transform(df[['education_score']])

# Investment proxy (usa_hey_shop as binary)
df['investment_proxy'] = df['usa_hey_shop'].astype(int)

df['score_buro_norm'] = scaler.fit_transform(df[['score_buro']])

# Combine with normalized score_buro
df['financial_sophistication'] = (
    df['education_score_norm'] + 
    df['investment_proxy'] + 
    df['score_buro_norm']
) / 3

print(df[['nivel_educativo', 'usa_hey_shop', 'score_buro', 'financial_sophistication']].head())

  nivel_educativo  usa_hey_shop  score_buro  financial_sophistication
0    Preparatoria          True         527                  0.583784
1    Preparatoria          True         714                  0.696096
2    Licenciatura          True         454                  0.651051
3        Posgrado         False         837                  0.658859
4    Preparatoria          True         533                  0.587387
